<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/t0c_unified_master_showcase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/T0C_Unified_Master_Showcase.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **T0C Unified Master Showcase**
### *Lattice Geometry Simulator for Thermal Routing, Coherence Mapping, and Phase‑Transition Analysis*

---

## **Overview**

This notebook provides a unified demonstration of the **T0C geometric routing model**, a framework for analyzing how small geometric detuning parameters influence:

- thermal routing  
- transparency vs. opacity  
- rigidity vs. flexibility  
- phase‑transition thresholds  
- coherence vs. dissipation  

The core engine is the **η‑selector**, a Gaussian resolver that converts:

- Δθ — geometric detuning  
- Δχ — cloud mismatch  
- Δf — frequency mismatch  

into a single routing probability **η**, which determines the dominant mode:

- **Straight‑Mode** — coherent propagation, transparency, efficient transport  
- **Loop‑Mode** — rigidity, inertial persistence  
- **Residue‑Mode** — phonon fog, heat, dissipation  

---

## **What This Notebook Demonstrates**

The interactive 4‑panel dashboard visualizes:

1. **Ice VII/X Acoustic Anomaly**  
   Routing probability η vs. pressure, highlighting the 64 GPa transition.

2. **Material Coherence Profiles**  
   η as a function of geometric angle for selected elements.

3. **Phase‑Clash Mapping**  
   Competition between rigidity (Loop) and transparency (Straight).

4. **Sodium Siphon COP Comparison**  
   Relative coherence gain and heat‑loss reduction under optimized routing.

A dedicated deep‑dive section explores sodium‑ion routing behavior, and a gradient‑analysis module demonstrates differentiability of the η‑selector.

---

## **Strategic Context**

This notebook supports the broader research program on:

- thermal‑mechanical resonance (300/n law)  
- conditional phase transitions (Bounce‑Gap)  
- geometric detuning thresholds (5% resolution gap)  
- high‑coherence energy systems  
- thermal‑routing engines for grid and battery applications  

It is intended as a **research‑grade demonstration tool** for materials science labs, energy‑systems researchers, and computational physics groups.

---

## **Document Version**
**v7.5 — Unified Showcase**  
**Primary Focus:** Coherence Optimization  
**Status:** Stable · Registry‑Driven

---


## 1. Engine Initialization  
### Core Geometry, Registry Loading, and η‑Selector Setup

This section initializes the T0C simulation engine, including:

- registry loading (constants, elements, molecules)  
- geometry utilities  
- η‑selector (Gaussian routing resolver)  
- autograd‑based gradient functions  
- global state for later modules  

All downstream visualizations and dashboards depend on this initialization block.


In [ ]:
# @title T0C Engine Setup — Initialization, Registry Loading, and η‑Selector

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, Tuple

import autograd.numpy as agnp
from autograd import grad
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

# -------------------------------------------------------------------
# Plot styling (dark theme)
# -------------------------------------------------------------------
plt.style.use("dark_background")
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "axes.edgecolor": "#444444",
        "grid.color": "#222222",
        "figure.facecolor": "#000000",
        "axes.facecolor": "#000000",
        "text.color": "#ffffff",
        "axes.labelcolor": "#ffffff",
        "xtick.color": "#ffffff",
        "ytick.color": "#ffffff",
    }
)

# -------------------------------------------------------------------
# Registry loading
# -------------------------------------------------------------------
REGISTRY_FILE = Path("T0C —  REGISTRY.json")

def load_t0c_registry(path: os.PathLike | str) -> Tuple[
    Dict[str, Any], Dict[str, Any], Dict[str, Any], Dict[str, Any]
]:
    """
    Load the T0C registry JSON and return:
    (full_data, constants, elements, molecules).
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Registry file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    meta = data.get("meta", {})
    constants = meta.get("constants", {})
    elements = data.get("elements", {})
    molecules = data.get("molecules", {})

    return data, constants, elements, molecules


def compute_element_dynamics(el: Dict[str, Any]) -> Dict[str, Any]:
    """
    Attach derived dynamics fields to an element record
    based on its gear assignment.
    """
    gear = el.get("gear_assignment", "metal")
    is_tetra = gear == "tetra"

    el = dict(el)
    el["eta_peak"] = 0.95 if is_tetra else 0.01
    el["flip_risk"] = 0.01 if is_tetra else 1.0
    return el


def calculate_ccz_p_metric(eta_values: Any) -> Any:
    """CCZ p‑metric: p = 1 − η."""
    return 1 - eta_values


def calculate_quantization_windows(max_val: float, n_start: int, n_end: int) -> np.ndarray:
    """
    Generate quantization window angles using max_val / n
    for integer n in [n_start, n_end], skipping n = 0.
    """
    ns = np.arange(n_start, n_end + 1, dtype=float)
    ns = ns[ns != 0]
    return max_val / ns


print("✅ Helper functions loaded.")


# -------------------------------------------------------------------
# Global state placeholders
# -------------------------------------------------------------------
T0C_REGISTRY: Dict[str, Any] | None = None
T0C_CONSTANTS: Dict[str, Any] | None = None
ELEMENTS: Dict[str, Any] | None = None
MOLECULES: Dict[str, Any] | None = None

grad_eta_wrt_theta = None
grad_eta_wrt_chi = None
grad_eta_wrt_f = None


# -------------------------------------------------------------------
# η‑selector (autograd‑compatible)
# -------------------------------------------------------------------
def eta_selector_autograd(
    delta_theta: Any,
    delta_chi: Any,
    delta_f: Any,
    constants: Dict[str, Any],
) -> Any:
    """
    Differentiable η‑selector:
    Gaussian in (Δθ, Δχ, Δf) with per‑axis sigmas from constants.
    """
    sig_t = float(constants.get("sigma_theta", 0.2))
    sig_c = float(constants.get("sigma_chi", 0.05))
    sig_f = float(constants.get("sigma_f", 0.01))

    dt = agnp.array(delta_theta)
    dc = agnp.array(delta_chi)
    df = agnp.array(delta_f)

    exponent = -(
        (dt ** 2) / (2 * sig_t ** 2)
        + (dc ** 2) / (2 * sig_c ** 2)
        + (df ** 2) / (2 * sig_f ** 2)
    )

    return agnp.clip(agnp.exp(exponent), 1e-9, 1.0)


# Public handle
eta_selector = eta_selector_autograd


# -------------------------------------------------------------------
# Initialization
# -------------------------------------------------------------------
try:
    T0C_REGISTRY, T0C_CONSTANTS, ELEMENTS, MOLECULES = load_t0c_registry(REGISTRY_FILE)

    # Enrich elements with derived dynamics
    ELEMENTS = {k: compute_element_dynamics(v) for k, v in ELEMENTS.items()}

    # Pre-bind constants into gradient functions
    _consts = T0C_CONSTANTS

    grad_eta_wrt_theta = grad(
        lambda dt, dc, df: eta_selector_autograd(dt, dc, df, _consts), argnum=0
    )
    grad_eta_wrt_chi = grad(
        lambda dt, dc, df: eta_selector_autograd(dt, dc, df, _consts), argnum=1
    )
    grad_eta_wrt_f = grad(
        lambda dt, dc, df: eta_selector_autograd(dt, dc, df, _consts), argnum=2
    )

    print(f"✅ T0C Engine Initialized — {len(ELEMENTS)} elements loaded.")
    print("✅ Differentiable η‑selector and gradient functions ready.")

except Exception as e:
    T0C_REGISTRY = None
    T0C_CONSTANTS = {}
    ELEMENTS = {}
    MOLECULES = {}
    print(f"❌ Registry load failed: {e}")


---

## 2. Unified T0C Dashboard  
### Interactive Visualization of Routing Behavior, Coherence Profiles, Phase‑Clash, and Siphon Efficiency

This section provides a unified 4‑panel dashboard demonstrating:

1. **Ice VII/X Acoustic Anomaly**  
   η vs. pressure, highlighting the 64 GPa transition.

2. **Material Coherence Profiles**  
   η as a function of geometric angle for selected elements.

3. **Phase‑Clash Mapping**  
   Rigidity (Loop) vs. Transparency (Straight) competition.

4. **Sodium Siphon COP Comparison**  
   Relative coherence gain and heat‑loss reduction.

All panels update interactively through ipywidgets.


In [ ]:
# @title Unified T0C Interactive Dashboard (4‑Panel Visualization)

def unified_t0c_dashboard(
    selected_materials=None,
    pressure_gpa: float = 64.0,
    p_scale_ice: float = 1.871,
    siphon_angle_offset: float = 0.0,
    std_freq_err: float = 0.05,
    opt_freq_err: float = 0.001,
):
    """
    Render the 4‑panel T0C dashboard:
      1) Ice VII/X acoustic anomaly
      2) Material coherence (η vs angle)
      3) Phase‑clash: rigidity vs transparency
      4) Sodium siphon COP comparison
    """
    if selected_materials is None:
        selected_materials = ['C', 'Al']

    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("T0C Unified Rendering Engine — Interactive Lab",
                 fontsize=18, color='white')
    colors = ['#00FFCC', '#FF3366', '#3399FF', '#FFFF00']

    # ---------------------------------------------------------------
    # Panel 1 — Ice VII/X Acoustic Anomaly
    # ---------------------------------------------------------------
    ax = axs[0, 0]
    pressure_range = np.linspace(0.0, 120.0, 300)

    delta_theta_0 = -4.97122
    align_p = 1.0 - 1.0 / (1.0 + pressure_range / p_scale_ice)
    delta_theta_p = delta_theta_0 * (1.0 - align_p)

    eta_ice = np.array([
        eta_selector(dt, 0.01, 0.001, T0C_CONSTANTS) for dt in delta_theta_p
    ])

    ax.plot(pressure_range, eta_ice, color='cyan', label='T0C η (H₂O)')
    ax.axvline(pressure_gpa, color='red', ls='--',
               label=f'Current P = {pressure_gpa:.1f} GPa')
    ax.axvline(64.0, color='white', ls=':', alpha=0.6,
               label='Predicted Transition (64 GPa)')
    ax.set_title("Ice VII/X Acoustic Anomaly")
    ax.set_xlabel("Pressure (GPa)")
    ax.set_ylabel("Routing Probability η")
    ax.legend()
    ax.grid(True, alpha=0.3)

    idx_p = int(np.argmin(np.abs(pressure_range - pressure_gpa)))
    detuning_at_p = float(delta_theta_p[idx_p])
    current_eta_ice = float(eta_ice[idx_p])

    # ---------------------------------------------------------------
    # Panel 2 — Material Coherence Profiles
    # ---------------------------------------------------------------
    ax = axs[0, 1]
    strain_range = np.linspace(30.0, 120.0, 200)

    for i, mat in enumerate(selected_materials):
        el = ELEMENTS.get(mat, {})
        theta_eq = el.get('theta_eq', 109.47)

        delta_theta = strain_range - theta_eq
        eta_mat = np.array([
            eta_selector(dt, 0.01, 0.001, T0C_CONSTANTS) for dt in delta_theta
        ])

        ax.plot(
            strain_range,
            eta_mat,
            label=f"{mat} η",
            color=colors[i % len(colors)],
            lw=2,
        )

    ax.axvline(T0C_CONSTANTS['theta_tetra'], color='white', ls=':',
               label='Tetra Lock')
    ax.axvline(T0C_CONSTANTS['theta_siphon'], color='gold', ls=':',
               label='Siphon Pivot')
    ax.set_title("Material Coherence (η vs Angle)")
    ax.set_xlabel("Geometric Angle θ (°)")
    ax.set_ylabel("Routing Probability η")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ---------------------------------------------------------------
    # Panel 3 — Phase‑Clash (Rigidity vs Transparency)
    # ---------------------------------------------------------------
    ax = axs[1, 0]

    siphon_theta_const = T0C_CONSTANTS['theta_siphon']
    siphon_distance = np.abs(strain_range - siphon_theta_const) / 90.0

    for i, mat in enumerate(selected_materials):
        el = ELEMENTS.get(mat, {})
        theta_eq = el.get('theta_eq', 109.47)

        delta_theta = strain_range - theta_eq
        eta_mat = np.array([
            eta_selector(dt, 0.01, 0.001, T0C_CONSTANTS) for dt in delta_theta
        ])

        rigidity = eta_mat ** 1.8
        transparency = eta_mat * (1.0 - siphon_distance)

        color = colors[i % len(colors)]
        ax.plot(strain_range, rigidity, '--', color=color,
                alpha=0.7, label=f"{mat} Rigidity")
        ax.plot(strain_range, transparency, color=color,
                label=f"{mat} Transparency")

    ax.fill_between(
        strain_range,
        0.9,
        1.0,
        color='green',
        alpha=0.15,
        label='Stability Zone',
    )
    ax.set_title("Phase‑Clash: Rigidity vs Transparency")
    ax.set_xlabel("Geometric Angle θ (°)")
    ax.set_ylabel("Mode Magnitude")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ---------------------------------------------------------------
    # Panel 4 — Sodium Siphon COP Comparison
    # ---------------------------------------------------------------
    ax = axs[1, 1]
    siphon_theta = T0C_CONSTANTS['theta_siphon'] + siphon_angle_offset

    systems = {
        'Standard': {
            'theta': 109.0,
            'f_err': std_freq_err,
            'color': '#444444',
        },
        'T0C‑Optimized': {
            'theta': siphon_theta,
            'f_err': opt_freq_err,
            'color': '#00FF00',
        },
    }

    for label, params in systems.items():
        delta_theta_sys = params['theta'] - siphon_theta
        eta_sys = float(
            eta_selector(delta_theta_sys, 0.01, params['f_err'], T0C_CONSTANTS)
        )
        cop = eta_sys * 2.065

        ax.bar(label, cop, color=params['color'], alpha=0.85, width=0.6)
        ax.text(label, cop + 0.05, f"{cop:.2f}",
                ha='center', color='white')

    ax.set_title("Sodium Siphon: Coefficient of Performance")
    ax.set_ylabel("COP (Relative Coherence Gain)")
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # ---------------------------------------------------------------
    # Live Status Summary
    # ---------------------------------------------------------------
    bounce_gap = T0C_CONSTANTS.get('bounce_gap', 0.141)
    status = (
        "COHERENCE LOCK (Transition)"
        if abs(detuning_at_p) <= bounce_gap
        else "RESIDUE / LOOP REGIME"
    )

    display(HTML(f"""
    <div style="background:#111; padding:12px; border:1px solid #444;
                border-radius:6px; color:#fff; font-family:monospace;">
        <b>Live T0C Status @ {pressure_gpa:.1f} GPa</b><br>
        Ice VII/X Detuning: {detuning_at_p:.4f}° |
        η = {current_eta_ice:.4f} |
        <span style="color:#0f0;">{status}</span>
    </div>
    """))


# Interactive controls
material_widget = widgets.SelectMultiple(
    options=list(ELEMENTS.keys()),
    value=['C', 'Al'],
    description='Materials:',
)

dashboard = widgets.interactive(
    unified_t0c_dashboard,
    selected_materials=material_widget,
    pressure_gpa=widgets.FloatSlider(
        value=64.0, min=0.0, max=120.0, step=0.5,
        description='Ice Pressure (GPa):'
    ),
    p_scale_ice=widgets.FloatSlider(
        value=1.871, min=0.1, max=5.0, step=0.001,
        description='Ice P_scale:'
    ),
    siphon_angle_offset=widgets.FloatSlider(
        value=0.0, min=-10.0, max=10.0, step=0.1,
        description='Siphon Offset:'
    ),
    std_freq_err=widgets.FloatSlider(
        value=0.05, min=0.001, max=0.1, step=0.001,
        description='Std Freq Error:'
    ),
    opt_freq_err=widgets.FloatSlider(
        value=0.001, min=0.0001, max=0.01, step=0.0001,
        description='Opt Freq Error:'
    ),
)

display(dashboard)


---

## 3. Sodium Siphon Deep Dive  
### Performance Audit, COP Sensitivity, and Heat‑Loss Analysis

This module provides a focused analysis of the sodium‑ion routing environment, comparing:

- Standard geometry  
- T0C‑optimized siphon geometry  

Outputs include:

- η (routing probability)  
- COP (coherence‑weighted performance)  
- Parasitic heat loss  
- Side‑by‑side bar charts  


In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

STANDARD_THETA = 109.0

def _compute_siphon_metrics(theta: float, siphon_theta: float, f_err: float):
    """Compute η, COP, and heat‑loss for a given siphon configuration."""
    eta = float(eta_selector(theta - siphon_theta, 0.01, f_err, T0C_CONSTANTS))
    nonlinear_cop_exponent = T0C_CONSTANTS.get('nonlinear_cop_exponent', 1.8) # Default to 1.8 if not found
    cop_base_multiplier = T0C_CONSTANTS.get('cop_base_multiplier', 2.065) # Default to 2.065 if not found
    default_heat_loss_factor = T0C_CONSTANTS.get('default_heat_loss_factor', 100.0) # Default to 100.0 if not found

    cop = (eta ** nonlinear_cop_exponent) * cop_base_multiplier
    heat_loss = (1.0 - eta) * default_heat_loss_factor
    return eta, cop, heat_loss


def sodium_siphon_deep_dive(
    siphon_angle_offset: float = 0.0,
    std_f_err: float = 0.05,
    opt_f_err: float = 0.001
):
    """Interactive audit of Sodium Siphon performance."""
    siphon_theta = T0C_CONSTANTS["theta_siphon"] + siphon_angle_offset

    systems = {
        "Standard": {"theta": STANDARD_THETA, "f_err": std_f_err},
        "T0C‑Optimized": {"theta": siphon_theta, "f_err": opt_f_err},
    }

    results = []
    for label, params in systems.items():
        eta, cop, heat_loss = _compute_siphon_metrics(
            theta=params["theta"],
            siphon_theta=siphon_theta,
            f_err=params["f_err"],
        )
        results.append(
            {"System": label, "η": eta, "COP": cop, "Heat Loss (J)": heat_loss}
        )

    df = pd.DataFrame(results)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    colors_cop = ["#666666", "#00ff88"]
    colors_heat = ["#ff4444", "#44aaff"]

    ax1.bar(df["System"], df["COP"], color=colors_cop, alpha=0.85)
    ax1.set_title("Coefficient of Performance")
    ax1.set_ylabel("COP (Relative Coherence Gain)")
    ax1.grid(axis="y", alpha=0.3)

    ax2.bar(df["System"], df["Heat Loss (J)"], color=colors_heat, alpha=0.85)
    ax2.set_title("Parasitic Heat Loss")
    ax2.set_ylabel("Joules")
    ax2.grid(axis="y", alpha=0.3)

    plt.suptitle("Sodium Siphon Performance Audit", fontsize=14, color="white")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    display(df.round(4))


deep_dive_widget = widgets.interactive(
    sodium_siphon_deep_dive,
    siphon_angle_offset=widgets.FloatSlider(
        value=0.0, min=-10.0, max=10.0, step=0.1, description="Siphon Offset:"
    ),
    std_f_err=widgets.FloatSlider(
        value=0.05, min=0.001, max=0.1, step=0.001, description="Std Freq Error:"
    ),
    opt_f_err=widgets.FloatSlider(
        value=0.001, min=0.0001, max=0.01, step=0.0001, description="Opt Freq Error:"
    ),
)

display(deep_dive_widget)

---

## 4. η‑Selector Gradient Analysis  
### Sensitivity of Routing Probability to Small Perturbations

This module demonstrates:

- differentiability of the η‑selector  
- sensitivity of η to Δθ, Δχ, and Δf  
- gradient magnitudes  
- perturbation‑response curves  

Useful for optimization, stability analysis, and parameter tuning.


In [ ]:
# @title η‑Selector Gradient Demonstration

def demonstrate_gradients(
    dt_val: float = 0.1,
    dc_val: float = 0.01,
    df_val: float = 0.001,
):
    """Demonstrate η sensitivity via autograd."""
    initial_eta = float(eta_selector(dt_val, dc_val, df_val, T0C_CONSTANTS))

    grad_theta = float(grad_eta_wrt_theta(dt_val, dc_val, df_val))
    grad_chi = float(grad_eta_wrt_chi(dt_val, dc_val, df_val))
    grad_f = float(grad_eta_wrt_f(dt_val, dc_val, df_val))

    print("--- η Selector Gradients ---")
    print(f"Input values: Δθ={dt_val:.4f}°, Δχ={dc_val:.4f}, Δf={df_val:.4f}")
    print(f"Initial η: {initial_eta:.6f}")
    print(f"Gradient wrt Δθ: {grad_theta:.6f}")
    print(f"Gradient wrt Δχ: {grad_chi:.6f}")
    print(f"Gradient wrt Δf: {grad_f:.6f}")
    print("\nInterpretation:")
    print("  • Positive gradient → η increases as the parameter increases")
    print("  • Negative gradient → η decreases as the parameter increases")
    print("  • Magnitude shows sensitivity strength")

    perturbation_range = np.linspace(-0.01, 0.01, 50)

    etas_theta = [
        eta_selector(dt_val + p, dc_val, df_val, T0C_CONSTANTS)
        for p in perturbation_range
    ]
    etas_chi = [
        eta_selector(dt_val, dc_val + p, df_val, T0C_CONSTANTS)
        for p in perturbation_range
    ]
    etas_f = [
        eta_selector(dt_val, dc_val, df_val + p, T0C_CONSTANTS)
        for p in perturbation_range
    ]

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"η Sensitivity to Small Perturbations (baseline η = {initial_eta:.4f})",
        fontsize=14,
        color="white",
    )

    axs[0].plot(perturbation_range, etas_theta, color="#00FFCC")
    axs[0].set_title("Δθ Perturbation")
    axs[0].grid(True, alpha=0.3)

    axs[1].plot(perturbation_range, etas_chi, color="#FF3366")
    axs[1].set_title("Δχ Perturbation")
    axs[1].grid(True, alpha=0.3)

    axs[2].plot(perturbation_range, etas_f, color="#3399FF")
    axs[2].set_title("Δf Perturbation")
    axs[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


widgets.interactive(
    demonstrate_gradients,
    dt_val=widgets.FloatSlider(
        value=0.1, min=-1.0, max=1.0, step=0.01, description="Δθ:"
    ),
    dc_val=widgets.FloatSlider(
        value=0.01, min=-0.1, max=0.1, step=0.001, description="Δχ:"
    ),
    df_val=widgets.FloatSlider(
        value=0.001, min=-0.01, max=0.01, step=0.0001, description="Δf:"
    ),
)
